In [175]:
import polars as pl
import polars.selectors as cs
import seaborn as sns
import matplotlib.pyplot as plt
from oauthlib.common import unquote

from src.features import build_features, ARCHIVELIST

#global settings
pl.Config.set_engine_affinity("streaming")
#pl.Config.set_tbl_rows(-1)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [176]:
df = pl.read_parquet("../data/parquets/sold_listings_20260830.parquet").lazy()
print(df.schema)
print(df.head())

Schema({'title': String, 'department': String, 'source': String, 'category': String, 'category_path': String, 'category_size': String, 'color': String, 'condition': String, 'size': String, 'styles': String, 'country_of_origin': String, 'price': Int32, 'sold_price': Int32, 'created_at': Datetime(time_unit='us', time_zone='UTC'), 'sold_at': Datetime(time_unit='us', time_zone='UTC'), 'cover_photo_url': String, 'location': String, 'seller_id': Int32, 'seller_total_transactions': Int32, 'seller_trusted': Boolean, 'seller_rating_average': Float64, 'seller_rating_count': Int32, 'followers_count': Int32, 'heat_score': Float64, 'photo_count': Int32, 'measurement_count': Int32, 'external_id': Int64, 'currency': String, 'local_image_path': String, 'image_download_status': String, 'original_price': Int32, 'scraped_at': Datetime(time_unit='us', time_zone='UTC'), 'designer_ids': List(Int32), 'designer_names': List(String), 'id': Int64})
naive plan: (run LazyFrame.explain(optimized=True) to see the o

C:\Users\mononoaware\AppData\Local\Temp\ipykernel_50048\3116410006.py:2: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print(df.schema)


In [177]:
#Null percentage count by column
nulls = (df.select(pl.all().null_count() / pl.len() * 100)
         .unpivot(variable_name="column_name", value_name="null_percentage")
         .filter(pl.col("null_percentage") > 0)
         .sort("null_percentage", descending=True))
print(nulls)


naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

SORT BY [descending: [true]] [col("null_percentage")]
  FILTER col("null_percentage") > 0.0
  FROM
    UNPIVOT on: [title, department, source, category, category_path, category_size, color, condition, size, styles, country_of_origin, price, sold_price, created_at, sold_at, cover_photo_url, location, seller_id, seller_total_transactions, seller_trusted, seller_rating_average, seller_rating_count, followers_count, heat_score, photo_count, measurement_count, external_id, currency, local_image_path, image_download_status, original_price, scraped_at, designer_ids, designer_names, id][], variable_name: column_name, value_name: null_percentage
      SELECT [((col("title").null_count() / len()) * 100.0), ((col("department").null_count() / len()) * 100.0), ((col("source").null_count() / len()) * 100.0), ((col("category").null_count() / len()) * 100.0), ((col("category_path").null_count() / len()) * 100.0), ((col("cat

In [178]:
#Checks how many listings have original_price == sold_price. original_price is derived from price_drops list so
og_is_sold = df.select(pl.col('original_price'), pl.col('sold_price')).filter(pl.col("sold_price") == pl.col('original_price')).count()
print(og_is_sold)


naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

SELECT [col("original_price").count(), col("sold_price").count()]
  FILTER col("sold_price") == col("original_price")
  FROM
    SELECT [col("original_price"), col("sold_price")]
      DF ["title", "department", "source", "category", ...]; PROJECT */35 COLUMNS


Restrict the eval window to 2025-08 onward where coverage is at least stable-ish. May be useful later, not in v1.

In [179]:
#First listing w styles was 1stl sold_at 2025
monthly = (df
    .group_by(pl.col('sold_at').dt.truncate('1mo').alias('month'))
    .agg(
        pl.len().alias('total'),
        (pl.col('styles') != '').sum().alias('has_styles'),
    )
    .with_columns((pl.col('has_styles') / pl.col('total')).alias('coverage'))
    .sort('month')
)
with pl.Config(tbl_rows=-1):
    print(monthly)

naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

SORT BY [col("month")]
   WITH_COLUMNS:
   [(col("has_styles") / col("total")).alias("coverage")] 
    AGGREGATE[maintain_order: false]
      [len().alias("total"), (col("styles") != "").sum().alias("has_styles")] BY [col("sold_at").dt.truncate(["1mo"]).alias("month")]
      FROM
      DF ["title", "department", "source", "category", ...]; PROJECT */35 COLUMNS


Restrict the eval window to 2025-06 -- 08 onward where coverage is at least stable-ish. May be useful later, not in v1.

In [180]:
monthly = (df
    .group_by(pl.col('sold_at').dt.truncate('1mo').alias('month'))
    .agg(
        pl.len().alias('total'),
        (pl.col('country_of_origin') != 'null').sum().alias('has_country'),
    )
    .with_columns((pl.col('has_country') / pl.col('total')).alias('coverage'))
    .sort('month')
)
with pl.Config(tbl_rows=-1):
    print(monthly)

naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

SORT BY [col("month")]
   WITH_COLUMNS:
   [(col("has_country") / col("total")).alias("coverage")] 
    AGGREGATE[maintain_order: false]
      [len().alias("total"), (col("country_of_origin") != "null").sum().alias("has_country")] BY [col("sold_at").dt.truncate(["1mo"]).alias("month")]
      FROM
      DF ["title", "department", "source", "category", ...]; PROJECT */35 COLUMNS


#Metrics for particular interest groups of designers - archive etc.

In [215]:
from src.features import ICARELIST
from src.train import train_w_folds, TEST_END, TRAIN_END
from dateutil.relativedelta import relativedelta
from datetime import datetime, UTC

df = pl.read_parquet("../data/parquets/sold_listings_20260901.parquet").lazy()
def train_eval(df: pl.LazyDataFrame):
    df = build_features(df)
    df = df.with_columns(
        pl.col('primary_designer').is_in(ARCHIVELIST).alias('archivelist'),
        pl.col('primary_designer').is_in(ICARELIST).alias('brands_icare')
    )
    start = datetime(2026, 2, 1, tzinfo=UTC)
    start = start - relativedelta(months=12)
    end = datetime(2026, 3, 1, tzinfo=UTC)
    segment = 'brands_icare' #'brands_icare'
    l = train_w_folds(df, TRAIN_END, TEST_END)
    for metric in l:
        print(metric)

train_eval(df)

Evaluating for everything


C:\Users\mononoaware\OneDrive\Documents\GitHub\GrailedDataAnalysis\.venv\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1389.725012 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 842481
[LightGBM] [Info] Number of data points in the train set: 4241419, number of used features: 222730
[LightGBM] [Info] Start training from score 4.418691
TOP 100 HIGHEST GAIN FEATURES
shape: (100, 2)
┌───────────────────┬───────────────┐
│ feature           ┆ importance    │
│ ---               ┆ ---           │
│ str               ┆ f64           │
╞═══════════════════╪═══════════════╡
│ primary_designer  ┆ 5.9583e6      │
│ path              ┆ 2.8342e6      │
│ vintage           ┆ 1.2356e6      │
│ 

How many listings with the same title are there?

In [243]:
not_unique_height = df.filter(~pl.col('title').is_unique()).collect().height
not_unique = df.filter(~pl.col('title').is_unique()).group_by(pl.col('title')).agg(pl.col('title').count().alias('count'))
print(not_unique)

1180865
